# Data Streams: The Stream Model

## Learning Objectives

1. **Define** the stream model and contrast it with batch processing
2. **Explain** the one-pass constraint and why it arises
3. **Describe** sliding window and decaying window semantics
4. **Identify** core stream query types

## The Stream Model

A **data stream** is a sequence of elements $a_1, a_2, a_3, \ldots$ that arrives in real time.

**Key constraints:**
- Elements arrive one at a time (or in small batches)
- We cannot store all elements — only a bounded summary
- We may only look at each element **once** (or a small constant number of times)
- Queries must be answered on the fly (low latency)

**Contrast with batch:**
| | Batch | Stream |
|--|--|--|
| Data size | Bounded, stored | Unbounded, transient |
| Passes | Multiple | One (or few) |
| Storage | Full dataset | Small summary |
| Query time | Offline | Real-time |

## Window Semantics

### Sliding Window
Only the **most recent $N$** elements are "active":
$$\text{Window}(t) = \{a_{t-N+1}, a_{t-N+2}, \ldots, a_t\}$$
Example: last 1 hour of click events.

### Landmark Window
All elements since a fixed start time (growing window).

### Decaying/Exponential Window
Each element has a **weight that decays** over time:
$$\hat{f}(t) = \sum_{i \leq t} c^{t-i} \cdot a_i, \quad c \in (0,1)$$
Recent elements matter more; old elements are "forgotten" smoothly.

**Half-life:** $\lambda_{1/2} = \log(2)/\log(1/c)$

In [ ]:
import math
from collections import deque

class SlidingWindowCounter:
    """Exact count of 1s in last N elements."""
    def __init__(self, N):
        self.N = N
        self.window = deque()
        self.count = 0
    def add(self, bit):
        self.window.append(bit)
        self.count += bit
        if len(self.window) > self.N:
            removed = self.window.popleft()
            self.count -= removed
    def query(self): return self.count

class DecayingCounter:
    """Exponentially decaying sum."""
    def __init__(self, c=0.99):
        self.c = c
        self.S = 0.0
    def add(self, value=1):
        self.S = self.c * self.S + value
    def query(self): return self.S

# Simulate a stream of 1s and 0s
import random; rng=random.Random(7)
stream = [rng.randint(0,1) for _ in range(500)]

N = 100
exact = SlidingWindowCounter(N)
decay = DecayingCounter(c=0.99)

snapshots = []
for t, bit in enumerate(stream):
    exact.add(bit); decay.add(bit)
    if t % 50 == 49:
        snapshots.append((t+1, exact.query(), decay.query()))

print(f"{'Time':>6} {'Exact (last 100)':>18} {'Decaying (c=0.99)':>20}")
for t, e, d in snapshots:
    print(f"{t:>6} {e:>18} {d:>20.2f}")

# Effective window of decaying counter
c = 0.99
eff_N = 1/(1-c)
print(f"
Decaying counter effective window ≈ 1/(1-c) = {eff_N:.0f} elements")

## Core Stream Query Types

| Query | Example | Challenge |
|-------|---------|-----------|
| **Count** | How many elements so far? | Trivial; test case for harder problems |
| **Distinct count** | How many unique users? | Exact = $O(n)$ space; approx = $O(\log n)$ |
| **Frequency** | How often does item $x$ appear? | Exact for popular items; approx for all |
| **Top-k / Heavy hitters** | What are the 10 most common items? | Must handle arbitrary item universe |
| **Windowed count of 1s** | How many 1s in last $N$ elements? | DGIM: $O(\log^2 N)$ space |
| **Membership** | Is element $x$ in the stream? | Bloom filter: one-sided errors |
| **Moments** | What is $\sum_i f_i^2$? | AMS sketch for 2nd moment |

## Space Complexity Landscape

For a stream with $n$ elements from a universe of size $m$:

| Problem | Exact space | Approximate space |
|---------|------------|-------------------|
| Count elements | $O(\log n)$ | Same |
| Count distinct | $O(m)$ or $O(n)$ | $O(\log m)$ (Flajolet-Martin) |
| Frequency of item $x$ | $O(m)$ | $O(1/\epsilon)$ (Count-Min) |
| Count 1s in window-$N$ | $O(N)$ | $O(\log^2 N)$ (DGIM) |
| Membership | $O(n)$ | $O(n)$ bits (Bloom filter) |
| 2nd moment | $O(m)$ | $O(1/\epsilon^2)$ (AMS) |

The stream model's goal: achieve $O(\text{polylog})$ space with provable approximation guarantees.